In [1]:
import torch
from torch import nn
from d2l import torch as d2l
def dropout_layer(X, dropout):
    assert 0 <= dropout <= 1
    # 在本情况中，所有元素都被丢弃
    if dropout == 1:
        return torch.zeros_like(X)
    # 在本情况中，所有元素都被保留
    if dropout == 0:
        return X
    mask = (torch.rand(X.shape) > dropout).float()
    return mask * X / (1.0 - dropout)

In [2]:
X= torch.arange(16, dtype = torch.float32).reshape((2, 8))
print(X)
print(dropout_layer(X, 0.))
print(dropout_layer(X, 0.5))
print(dropout_layer(X, 1.))

tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])
tensor([[ 0.,  2.,  0.,  6.,  8., 10., 12., 14.],
        [ 0., 18.,  0., 22., 24.,  0.,  0., 30.]])
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


定义模型参数

In [3]:
num_inputs, num_outputs, num_hiddens1, num_hiddens2 = 784, 10, 256, 256

定义模型

In [7]:
dropout1, dropout2 = 0.2, 0.5
class Net(nn.Module):
    def __init__(self, num_inputs, num_outputs, num_hiddens1, num_hiddens2,
                is_training = True):
        super(Net, self).__init__()
        self.num_inputs = num_inputs
        self.training = is_training
        self.lin1 = nn.Linear(num_inputs, num_hiddens1)
        self.lin2 = nn.Linear(num_hiddens1, num_hiddens2)
        self.lin3 = nn.Linear(num_hiddens2, num_outputs)
        self.relu = nn.ReLU()
    
    def forward(self, X):
        H1 = self.relu(self.lin1(X.reshape((-1, self.num_inputs))))
        if self.training == True:
            H1 = dropout_layer(H1, dropout1)
        H2 = self.relu(self.lin2(H1))
        if self.training == True:
            H2 = dropout_layer(H2, dropout2)
        out = self.lin3(H2)
        return out

net = Net(num_inputs, num_outputs, num_hiddens1, num_hiddens2)

训练和测试

In [8]:
def evaluate_accuracy(net, data_iter):
    """计算在指定数据集上模型的精度"""
    net.eval()
    metric_sum, metric_num = 0, 0
    with torch.no_grad():
        for X, y in data_iter:
            metric_sum += (net(X).argmax(dim=1) == y).float().sum().item()
            metric_num += y.numel()
    return metric_sum / metric_num

def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):
    """训练模型"""
    for epoch in range(num_epochs):
        net.train()
        loss_sum, acc_sum, n = 0.0, 0.0, 0
        for X, y in train_iter:
            # 前向传播
            y_hat = net(X)
            l = loss(y_hat, y).mean()
            
            # 反向传播和优化
            updater.zero_grad()
            l.backward()
            updater.step()
            
            # 记录损失和精度
            with torch.no_grad():
                metric = (l * len(y)).sum().item()
                loss_sum += metric
                acc_sum += (y_hat.argmax(1) == y).sum().item()
                n += y.numel()
        
        # 计算测试精度
        test_acc = evaluate_accuracy(net, test_iter)
        
        # 打印结果
        train_loss = loss_sum / n
        train_acc = acc_sum / n
        print(f'epoch {epoch + 1}, loss {train_loss:.4f}, train acc {train_acc:.3f}, test acc {test_acc:.3f}')
    
    # 恢复训练模式
    net.train()

In [10]:
num_epochs, lr, batch_size = 10, 0.5, 256
loss = nn.CrossEntropyLoss(reduction='none')
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)
trainer = torch.optim.SGD(net.parameters(), lr=lr)
train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

epoch 1, loss 0.8931, train acc 0.670, test acc 0.743
epoch 2, loss 0.5215, train acc 0.810, test acc 0.785
epoch 3, loss 0.4587, train acc 0.832, test acc 0.804
epoch 4, loss 0.4274, train acc 0.844, test acc 0.852
epoch 5, loss 0.3992, train acc 0.853, test acc 0.829
epoch 6, loss 0.3851, train acc 0.859, test acc 0.854
epoch 7, loss 0.3689, train acc 0.864, test acc 0.859
epoch 8, loss 0.3551, train acc 0.870, test acc 0.861
epoch 9, loss 0.3450, train acc 0.873, test acc 0.852
epoch 10, loss 0.3347, train acc 0.877, test acc 0.868
